In [1]:
# %pip install pandas numpy matplotlib seaborn scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
training = pd.read_csv("../training_data.csv", encoding="latin1")
test = pd.read_csv("../test_data.csv", encoding="latin1")

In [3]:
# Dropping unique values

training.drop('city_name', axis=1, inplace=True)
test.drop('city_name', axis=1, inplace=True)

training.drop('AVERAGE_PRECIPITATION', axis=1, inplace=True)
test.drop('AVERAGE_PRECIPITATION', axis=1, inplace=True)

In [4]:
training['LUMINOSITY'] = pd.factorize(training['LUMINOSITY'])[0] + 1
test['LUMINOSITY'] = pd.factorize(test['LUMINOSITY'])[0] + 1

In [5]:
cloudiness_training = pd.get_dummies(training['AVERAGE_CLOUDINESS'], drop_first=True)
cloudiness_test = pd.get_dummies(test['AVERAGE_CLOUDINESS'], drop_first=True)
rain_training = pd.get_dummies(training['AVERAGE_RAIN'], drop_first=True)
rain_test = pd.get_dummies(test['AVERAGE_RAIN'], drop_first=True)

training.drop(['AVERAGE_CLOUDINESS', 'AVERAGE_RAIN'], axis=1, inplace=True)
training = pd.concat([training,cloudiness_training, rain_training], axis=1)

test.drop(['AVERAGE_CLOUDINESS', 'AVERAGE_RAIN'], axis=1, inplace=True)
test = pd.concat([test,cloudiness_test, rain_test], axis=1)

In [6]:
p_med = training.loc[training['AVERAGE_ATMOSP_PRESSURE'] > 900, 'AVERAGE_ATMOSP_PRESSURE'].median()
training.loc[training['AVERAGE_ATMOSP_PRESSURE'] < 900, 'AVERAGE_ATMOSP_PRESSURE'] = p_med
test.loc[test['AVERAGE_ATMOSP_PRESSURE'] < 900, 'AVERAGE_ATMOSP_PRESSURE'] = p_med

In [7]:
training = training.dropna(subset=['record_date'])
training['record_date'] = pd.to_datetime(training['record_date'], format = '%Y-%m-%d %H:%M:%S', errors = 'coerce')
assert training['record_date'].isnull().sum() == 0, 'missing record date'
print(training['record_date'].head())

test = test.dropna(subset=['record_date'])
test['record_date'] = pd.to_datetime(test['record_date'], format = '%Y-%m-%d %H:%M:%S', errors = 'coerce')
assert test['record_date'].isnull().sum() == 0, 'missing record date'
print(test['record_date'].head())

0   2019-08-29 07:00:00
1   2018-08-10 14:00:00
2   2019-09-01 16:00:00
3   2019-02-26 11:00:00
4   2019-06-06 12:00:00
Name: record_date, dtype: datetime64[ns]
0   2019-02-13 23:00:00
1   2018-11-28 20:00:00
2   2018-08-14 05:00:00
3   2019-07-06 17:00:00
4   2018-10-15 06:00:00
Name: record_date, dtype: datetime64[ns]


In [8]:
training['record_date_year'] = training['record_date'].dt.year
training['record_date_month'] = training['record_date'].dt.month
training['record_date_day'] = training['record_date'].dt.day
training['record_date_hour'] = training['record_date'].dt.hour
training['record_date_minute'] = training['record_date'].dt.minute
training['record_date_second'] = training['record_date'].dt.second
training.head()

test['record_date_year'] = test['record_date'].dt.year
test['record_date_month'] = test['record_date'].dt.month
test['record_date_day'] = test['record_date'].dt.day
test['record_date_hour'] = test['record_date'].dt.hour
test['record_date_minute'] = test['record_date'].dt.minute
test['record_date_second'] = test['record_date'].dt.second
test.head()

,record_date,AVERAGE_FREE_FLOW_SPEED,AVERAGE_TIME_DIFF,AVERAGE_FREE_FLOW_TIME,LUMINOSITY,AVERAGE_TEMPERATURE,AVERAGE_ATMOSP_PRESSURE,AVERAGE_HUMIDITY,AVERAGE_WIND_SPEED,céu claro,...,chuva moderada,chuvisco fraco,trovoada com chuva,trovoada com chuva leve,record_date_year,record_date_month,record_date_day,record_date_hour,record_date_minute,record_date_second
0,2019-02-13 23:00:00,39.2,0.0,91.0,1,8.0,1026.0,71.0,1.0,True,...,False,False,False,False,2019,2,13,23,0,0
1,2018-11-28 20:00:00,42.5,12.2,76.8,1,11.0,1020.0,93.0,4.0,False,...,False,False,False,False,2018,11,28,20,0,0
2,2018-08-14 05:00:00,45.9,0.0,86.3,1,14.0,1017.0,93.0,0.0,False,...,False,False,False,False,2018,8,14,5,0,0
3,2019-07-06 17:00:00,33.2,51.7,89.9,2,22.0,1016.0,77.0,4.0,False,...,False,False,False,False,2019,7,6,17,0,0
4,2018-10-15 06:00:00,44.0,3.5,85.5,1,12.0,1004.0,100.0,9.0,False,...,False,False,False,False,2018,10,15,6,0,0


In [9]:
# Dropping unique date values

training.drop('record_date_minute', axis=1, inplace=True)
training.drop('record_date_second', axis=1, inplace=True)
training.drop('record_date', axis=1, inplace=True)
#training.dropna(inplace=True)
training.head()

test.drop('record_date_minute', axis=1, inplace=True)
test.drop('record_date_second', axis=1, inplace=True)
test.drop('record_date', axis=1, inplace=True)
test.dropna(inplace=True)
test.head()

,AVERAGE_FREE_FLOW_SPEED,AVERAGE_TIME_DIFF,AVERAGE_FREE_FLOW_TIME,LUMINOSITY,AVERAGE_TEMPERATURE,AVERAGE_ATMOSP_PRESSURE,AVERAGE_HUMIDITY,AVERAGE_WIND_SPEED,céu claro,céu limpo,...,chuva fraca,chuva leve,chuva moderada,chuvisco fraco,trovoada com chuva,trovoada com chuva leve,record_date_year,record_date_month,record_date_day,record_date_hour
0,39.2,0.0,91.0,1,8.0,1026.0,71.0,1.0,True,False,...,False,False,False,False,False,False,2019,2,13,23
1,42.5,12.2,76.8,1,11.0,1020.0,93.0,4.0,False,False,...,False,False,False,False,False,False,2018,11,28,20
2,45.9,0.0,86.3,1,14.0,1017.0,93.0,0.0,False,False,...,False,False,False,False,False,False,2018,8,14,5
3,33.2,51.7,89.9,2,22.0,1016.0,77.0,4.0,False,False,...,False,False,False,False,False,False,2019,7,6,17
4,44.0,3.5,85.5,1,12.0,1004.0,100.0,9.0,False,False,...,True,False,False,False,False,False,2018,10,15,6


In [10]:
# Preparar X_train e y_train
X_train_full = training.drop('AVERAGE_SPEED_DIFF', axis=1)
y_train_full = training['AVERAGE_SPEED_DIFF']

# X_test é o test completo (não tem AVERAGE_SPEED_DIFF)
X_test = test.copy()

print("Antes do alinhamento:")
print(f"X_train: {X_train_full.shape}")
print(f"X_test: {X_test.shape}")
print(f"X_train: {X_train_full.shape}")
print(f"X_test: {X_test.shape}")

# Adicionar colunas que existem no training mas não no test
missing_cols = set(X_train_full.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0
    print(f"Adicionada coluna '{col}' ao test")

# Remover colunas que existem no test mas não no training
extra_cols = set(X_test.columns) - set(X_train_full.columns)
if len(extra_cols) > 0:
    X_test.drop(list(extra_cols), axis=1, inplace=True)
    print(f"Removidas {len(extra_cols)} colunas extra do test")

# Reordenar colunas do test para ficarem na mesma ordem que o training
X_test = X_test[X_train_full.columns]

print("\nDepois do alinhamento:")
print(f"X_train: {X_train_full.shape}")
print(f"X_test: {X_test.shape}")
print(f"Colunas coincidem: {list(X_train_full.columns) == list(X_test.columns)}")

Antes do alinhamento:
X_train: (6812, 32)
X_test: (1500, 28)
X_train: (6812, 32)
X_test: (1500, 28)
Adicionada coluna 'chuva forte' ao test
Adicionada coluna 'chuva de intensidade pesado' ao test
Adicionada coluna 'chuvisco e chuva fraca' ao test
Adicionada coluna 'chuva de intensidade pesada' ao test

Depois do alinhamento:
X_train: (6812, 32)
X_test: (1500, 32)
Colunas coincidem: True


In [11]:
y_train_full.fillna("None", inplace=True) # muito importante

In [12]:
y_train_full.value_counts()

AVERAGE_SPEED_DIFF
None         2200
Medium       1651
Low          1419
High         1063
Very_High     479
Name: count, dtype: int64

In [ ]:
mapper = {"None": 0, "Low": 1, "Medium": 2, "High": 3, "Very_High": 4}
y_train_full = y_train_full.map(mapper)
y_train_full.value_counts()

AVERAGE_SPEED_DIFF
0    2200
2    1651
1    1419
3    1063
4     479
Name: count, dtype: int64

In [14]:
from sklearn.model_selection import train_test_split

# Dividir dados para validação
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, 
    y_train_full,
    test_size=0.3,      # 30% para validação
    random_state=42,
    stratify=y_train_full  # Mantém proporções das classes
)

print(f"\nTreino: {X_train.shape}")
print(f"Validação: {X_val.shape}")
print(f"Test: {X_test.shape}")


Treino: (4768, 32)
Validação: (2044, 32)
Test: (1500, 32)


In [15]:
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier


gb_clf = GradientBoostingClassifier(
    n_estimators=300,       # Número de árvores
    learning_rate=0.05,     # Taxa de aprendizagem (menor = melhor generalização)
    max_depth=6,            # Profundidade intermédia
    subsample=0.8,          # Usa 80% dos dados por árvore (reduz overfitting)
    random_state=42
)
rf_clf = RandomForestClassifier(
    n_estimators=300, 
    max_depth=15,           # Profundidade controlada
    random_state=42, 
    n_jobs=-1               # Usar todos os processadores
)

voting_clf = VotingClassifier(
    estimators=[
        ('gb', gb_clf),
        ('rf', rf_clf)
    ],
    voting='soft'  # Usa a média das probabilidades, não apenas o voto final
)

print("A treinar o Ensemble (isto vai demorar um pouco)...")
voting_clf.fit(X_train_full, y_train_full)

A treinar o Ensemble (isto vai demorar um pouco)...


,estimators,"[('gb', ...), ('rf', ...)]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,loss,'log_loss'
,learning_rate,0.05
,n_estimators,300
,subsample,0.8
,criterion,'friedman_mse'


In [16]:
from sklearn.metrics import classification_report
# Avaliação no conjunto de treino
y_pred_train = voting_clf.predict(X_train_full)
print("\n--- Resultados no Treino ---")
print(classification_report(y_train_full, y_pred_train))
print(f"Accuracy no Treino: {accuracy_score(y_train_full, y_pred_train):.4f}")

# Avaliação no conjunto de validação
y_pred = voting_clf.predict(X_val)
print("\n--- Resultados na Validação ---")
print(classification_report(y_val, y_pred))
print(f"Accuracy na Validação: {accuracy_score(y_val, y_pred):.4f}")


--- Resultados no Treino ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2200
           1       1.00      1.00      1.00      1419
           2       1.00      1.00      1.00      1651
           3       1.00      1.00      1.00      1063
           4       1.00      1.00      1.00       479

    accuracy                           1.00      6812
   macro avg       1.00      1.00      1.00      6812
weighted avg       1.00      1.00      1.00      6812

Accuracy no Treino: 0.9993

--- Resultados na Validação ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       660
           1       1.00      1.00      1.00       426
           2       1.00      1.00      1.00       495
           3       1.00      1.00      1.00       319
           4       1.00      1.00      1.00       144

    accuracy                           1.00      2044
   macro avg       1.00      1.00      1.

In [17]:
print("\nA re-treinar com o dataset completo para submissão...")
voting_clf.fit(X_train_full, y_train_full)

# Previsão no Teste
test_predictions = voting_clf.predict(X_test)

# Criar CSV
inv_map = {0: "None", 1: "Low", 2: "Medium", 3: "High", 4: "Very_High"}
final_predictions = pd.Series(test_predictions).map(inv_map)

submission = pd.DataFrame({
    "RowId": range(1, len(final_predictions) + 1),
    "Speed_Diff": final_predictions
})


A re-treinar com o dataset completo para submissão...


In [18]:
nome_ficheiro = "../Predictions/Ensemble_GB_RF_2.csv"
submission.to_csv(nome_ficheiro, index=False)
print(f"Ficheiro '{nome_ficheiro}' criado com sucesso!")

Ficheiro '../Predictions/Ensemble_GB_RF_2.csv' criado com sucesso!
